In [ ]:
from dataclasses import dataclass
import os
from typing import Optional
from openai import OpenAI
import json
from pydantic import BaseModel
from typing import Literal
from dotenv import load_dotenv

load_dotenv()

@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests across all inference providers"""
    name:str
    env_var:str
    is_free:bool
    base_url: Optional[str]
    model:str

PROVIDERS = [
   Provider("OpenAI","OPENAI_API_KEY",True,None,"gpt-4o-mini"),    
   Provider("Groq","GROQ_API_KEY",True,"https://api.groq.com/openai/v1","openai/gpt-oss-120b"),
   
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")

def build_client(provider: Provider) -> OpenAI:


    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )


def have_any_key()->bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")    


def llm_reply(prompt:str)->str:
    provider = select_provider()
    print(provider)
    client = build_client(provider)
    result = client.chat.completions.create(
        model = provider.model,
        max_tokens=200,
        messages=[
            {
                "role":"user",
                "content":prompt,
            }
        ]
    )

    return result.choices[0].message.content


class CotStep(BaseModel):
    """
    A step in the COT process
    """
    content:str
    step_type:Literal["THINKING","FINAL_OUTPUT"] #Thinking -> Plan


def parse_cot_step(raw_reply:str) -> CotStep | str:
    """Parse a raw reply into a CotStep"""
    try:
        parsed = json.loads(raw_reply) # returning a python dict
        return CotStep(**parsed) #converting python dict to a CotStep object
    except:
        return f"Invalid JSON:{raw_reply}"

SYSTEM_PROMPT = """
You are an expert assistant that solves user queries one step at a time.

You have two possible step types:

- THINKING: one short reasoning step toward solving the problem.
- FINAL_OUTPUT: the final answer to the user's question.

IMPORTANT RULES:

1. Each API response MUST contain exactly ONE JSON object.
2. NEVER return multiple JSON objects in a single response.
3. NEVER return an array of JSON objects.
4. NEVER return markdown, code fences, or any text outside the JSON object.
5. Return exactly one step per API response.
6. If more reasoning is needed, return a THINKING step.
7. If the answer is ready, return a FINAL_OUTPUT step.
8. The "step_type" value must be exactly either "THINKING" or "FINAL_OUTPUT".
9. The "content" value must be a string.

Use exactly this JSON format:

{
    "content": "your next reasoning step or final answer",
    "step_type": "THINKING"
}

OR, when the answer is ready:

{
    "content": "your final answer",
    "step_type": "FINAL_OUTPUT"
}

Example:

User question:
Roger has 5 tennis balls. He buys 2 cans of tennis balls. Each can contains 3 tennis balls. How many tennis balls does Roger have now?

For the first API response, return only:

{
    "step_type": "THINKING",
    "content": "Roger starts with 5 tennis balls."
}

On the next API call, return only the next step, for example:

{
    "step_type": "THINKING",
    "content": "Two cans contain 2 × 3 = 6 tennis balls."
}

On a later API call, when the answer is ready, return only:

{
    "step_type": "FINAL_OUTPUT",
    "content": "Roger has 11 tennis balls now."
}

Remember: ONE API CALL = ONE JSON OBJECT = ONE STEP.
"""


def llm_json_reply(messages:list[dict])->str:
    provider = select_provider()
    client = build_client(provider)

    kwargs: dict = {
        "model":provider.model,
        "max_tokens":400,
        "messages":messages,
    }

    result = client.chat.completions.create(**kwargs)
    return result.choices[0].message.content

In [ ]:
WEATHER_DB = {
    "london":{"celcius":22,"sky":"cloudy"},
    "paris":{"celcius":24,"sky":"sunny"},
    "tallin":{"celcius":28,"sky":"sunny"},
    "moscow":{"celcius":18,"sky":"rainy"}
}

In [ ]:
def lookup_weather(location : str) -> str:
    """Look up the weather for a location"""
    record = WEATHER_DB.get(location.lower())
    print(f"looking up weather for {location.lower()}")
    print(record)

    if record is None:
        return f"Sorry , I dont know the weather in {location}"
    return f"The weather in {location} is {record['celcius']} and {record['sky']}"

In [ ]:
weather_schema = {
    "type":"function",
    "function":{
        "name":"lookup_weather",
        "description":"Look up the weather for a location",
        "parameters":{
            "type":"object",
            "properties":{
                "location":{"type":"string","description":"The location to look up the weather for."}
            }
        },
        "required":["location"]
    }
}# this schema will be given to the llm


TOOLS = {
    "lookup_weather":lookup_weather,
    "stock_price_checker":None
}

In [ ]:
def ask_llm_with_tool(prompt:str,*,max_tokens:int = 1000)-> str:
    """
    Call the llm with a tool call.
    """

    provider = select_provider()
    client = build_client(provider)
    result = client.chat.completions.create(
        model = provider.model,
        max_tokens = max_tokens,
        messages = [
            {"role":"user","content":prompt}
            ],
        tools = [weather_schema],
    )
    print(result.choices)
    return result.choices[0].message


In [ ]:
msg = ask_llm_with_tool("What is the weather in london compared to paris?")

In [ ]:
if msg.tool_calls:
    print("tool call detected")
    print("Total tool calls",len(msg.tool_calls))
    for tool_call in msg.tool_calls:
        print("Tool call name",tool_call.function.name)
        print("Tool call arguments",tool_call.function.arguments)
        print("Tool call parsed",json.loads(tool_call.function.arguments))
        city = json.loads(tool_call.function.arguments)['location']
        print(f"Calling tool for {city}")
        tool = TOOLS[tool_call.function.name] # this will give us the function
        result = tool(city)
        print(result)
        print("-"*100)


In [ ]:
from email import message


def ask_llm_with_tool(prompt: str,*,max_tokens:int = 400) -> str:
    """Call the LLM with a tool call"""

    provider = select_provider()
    client = build_client(provider)
    messages = [
        {"role" : "user" , "content":prompt}
    ]

    while True:
        response = client.chat.completions.create(
            model = provider.model,
            max_tokens = max_tokens,
            messages = messages,
            tools = [weather_schema]
        )

        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content
        
        messages.append({"role":"assistant","content":msg.content,"tool_calls":msg.tool_calls})

        #llm is asking us to call a tool

        for tool_call in msg.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            tool_call_id = tool_call.id

            if tool_name not in TOOLS:
                raise ValueError(f"Tool {tool_name} not found")
            
            tool = TOOLS[tool_name] #actual function
            result = tool(**tool_args) # -> lookup_weather(location = "london")

            messages.append(
                {"role":"tool","tool_call_id":tool_call_id,"content":result}
            )

        print("All tool calls done,going back to LLM")


        
    




In [ ]:
ask_llm_with_tool("What is the weather like in London compared to Paris? ")